# Image Classification for AutoVision Ai

### AutoVision AI

**AutoVision AI** is an AI-powered image classification and prediction platform built using **Python, TensorFlow, and Django**. The project allows users to upload an image, processes the image using a trained deep learning model, and returns the predicted class.

The main goal of AutoVision AI is to create a reusable system where different image datasets can be used for training and the trained model can be integrated into a web application for real-time prediction.

The project follows a complete machine learning workflow:

**Dataset → Data Preprocessing → Model Training → Model Evaluation → Model Saving → Django Web Application → Image Prediction**

The project is designed with a modular structure so that new datasets and deep learning models can be added easily in the future.


In [ ]:
# ==============================
# Data Manipulation
# ==============================

import pandas as pd          # Used for loading, cleaning, and analyzing tabular data
import numpy as np           # Used for numerical operations and array manipulation


# ==============================
# Data Visualization
# ==============================

import matplotlib.pyplot as plt   # Used for creating graphs, plots, and visualizing results
import seaborn as sns             # Used for statistical data visualization
from PIL import Image             # Used for opening, processing, and manipulating images


# ==============================
# Deep Learning / Neural Networks
# ==============================

import tensorflow as tf                    # Main framework for building and training deep learning models
import keras                               # High-level API for building neural networks
from tensorflow.keras import layers, models  # Used to create neural network layers and model architectures


# ==============================
# File and Path Management
# ==============================

import pathlib                    # Used for handling file and directory paths
import PIL
import cv2


# ==============================
# Warning Management
# ==============================

import warnings                   # Used to control and suppress Python warnings
warnings.filterwarnings('ignore') # Hides warning messages to keep the output clean

In [ ]:
data_url="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz" # dataset Url
data_dir=tf.keras.utils.get_file(
    'flower_photos',
    origin=data_url,
    cache_dir='.',
    untar=True

)

In [ ]:
data_dir=pathlib.Path(data_dir)/'flower_photos'
data_dir

In [ ]:
image_count=len(list(data_dir.glob('*/*.jpg')))
print(f'Total No of Images :{image_count}')

In [ ]:
images_paths=list(data_dir.glob('*/*.jpg'))
print(f'5 Images paths :\n {images_paths[:5]}')

# printing images using pillow

In [ ]:
PIL.Image.open(images_paths[0])

In [ ]:
PIL.Image.open(images_paths[1])

In [ ]:
dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,                 # Load images from the dataset directory
    batch_size=32,            # Process 32 images per batch
    image_size=(180, 180),    # Resize all images to 180 × 180 pixels
    shuffle=True              # Shuffle images before training
)

In [ ]:
class_name=dataset.class_names
print(f'class names :{class_name}')

In [ ]:
# image read using opencv from image path
img=cv2.imread(images_paths[0])
img.shape

In [ ]:
flowers_images_dict={
    'daisy':list(data_dir.glob('daisy/*')),
    'dandelion':list(data_dir.glob('dandelion/*')),
    'roses':list(data_dir.glob('roses/*')),
    'sunflowers':list(data_dir.glob('sunflowers/*')),
    'tulips':list(data_dir.glob('tulips/*'))
}
flowers_labels_dict={
    'daisy':0,
    'dandelion':1,
    'roses':2,
    'sunflowers':3,
    'tulips':4
}

In [ ]:
for name in class_name:
  print(len(flowers_images_dict[name]))

In [ ]:
plt.figure(figsize=(6,3))
for flower,images in flowers_images_dict.items():
  for i in range(1,7):
      img=cv2.imread(images[i])
      img_resized=cv2.resize(img,(180,180))
      plt.subplot(2,3,i)
      plt.imshow(img_resized)
      plt.title(flower)
      plt.axis('off')

In [ ]:
x,y=[],[]
for flower,images in flowers_images_dict.items():
  for image in images:
    img=cv2.imread(image)
    img_resized=cv2.resize(img,(180,180))
    x.append(img_resized)
    y.append(flowers_labels_dict[flower])

In [ ]:
X=np.array(x)
Y=np.array(y)
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.25,shuffle=True)
x_train_scaled=x_train/255
x_test_scaled=x_test/255


In [ ]:
print(len(x_train_scaled))

In [ ]:
# building cnn model
model=keras.Sequential([

    # CNN
    layers.Input(shape=(180,180,3)),

    layers.Conv2D(16,(3,3),padding='same',activation='relu'),
    layers.MaxPool2D(),

    layers.Conv2D(32,(3,3),padding='same',activation='relu'),
    layers.MaxPool2D(),

    layers.Conv2D(64,(3,3),padding='same',activation='relu'),
    layers.MaxPool2D(),

    layers.Flatten(),

    layers.Dense(64,activation='relu'),

    layers.Dense(len(class_name),activation='softmax')

])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
y_train

In [ ]:
model.fit(x_train_scaled,y_train,epochs=5)

In [ ]:
model.evaluate(x_test_scaled,y_test)

In [ ]:
res=model.predict(x_test_scaled)
y_predict=[int(np.argmax(i)) for i in res]

In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test,y_predict))